# Sample size for model development

## Background

Two papers by the same group underpin this notebook, and both are cited because they number
their criteria differently.

> **Riley RD, Snell KIE, Ensor J, Burke DL, Harrell FE Jr, Moons KGM, Collins GS.** Minimum
> sample size for developing a multivariable prediction model: Part II - binary and
> time-to-event outcomes. *Statistics in Medicine* 2019;38:1276-1296. doi:10.1002/sim.7992

The **methods paper**. It derives the closed-form equations used below and proposes three
requirements: (i) small optimism in predictor effects, defined by a uniform shrinkage factor
S >= 0.9; (ii) a difference of no more than 0.05 between the apparent and optimism-adjusted
Nagelkerke R-squared; and (iii) precise estimation of the overall outcome risk.

> **Riley RD, Ensor J, Snell KIE, Harrell FE Jr, Martin GP, Reitsma JB, Moons KGM, Collins GS,
> van Smeden M.** Calculating the sample size required for developing a clinical prediction
> model. *BMJ* 2020;368:m441. doi:10.1136/bmj.m441

The **practical guide**. It restates those requirements as a four-step procedure for binary
outcomes, labels them **B1-B4** (the labels used throughout this notebook), and adds a further
criterion on mean absolute prediction error taken from:

> **van Smeden M, Moons KGM, de Groot JAH, et al.** Sample size for binary logistic prediction
> models: beyond events per variable criteria. *Statistical Methods in Medical Research*
> 2019;28:2455-2474. doi:10.1177/0962280218784726

### Mapping between the two papers

The numbering does **not** correspond, which is a common source of confusion: criterion (i) of
the 2019 paper is B3 of the 2020 paper, and criterion (iii) is B1.

| Label (BMJ 2020) | What it requires | Equivalent in Stat Med 2019 | Formula |
|---|---|---|---|
| **B1** | Overall outcome proportion estimated to within +/- 0.05 | criterion (iii) | BMJ fig 1 = 2019 eq (27) |
| **B2** | Mean absolute prediction error <= 0.05 | *not in the 2019 paper* | BMJ fig 2, from van Smeden et al. |
| **B3** | Uniform shrinkage factor S >= 0.9, i.e. <= 10% overfitting | criterion (i) | BMJ fig 3 = 2019 eq (11) |
| **B4** | Optimism in apparent Nagelkerke R-squared <= 0.05 | criterion (ii) | BMJ fig 5 = 2019 eq (26) |

All four must be satisfied; the required sample size is the largest of the four.

## This study

We are developing a **binary prediction model** for death within 30 days of a spinal trauma
consultation:

- **N = 1,486** included patients
- **E = 198** deaths within 30 days (event proportion phi = 13.3%)
- **P = 25** candidate predictor parameters, read directly from `config/pipeline.yaml`

### Two points on how this framework is applied here

**P counts every candidate parameter.** The model uses a ridge (L2) penalty, which shrinks the
coefficients but sets none of them to zero, so every candidate parameter stays in the final model.
Riley et al. are explicit that P must be the total number of parameters for all predictors
*screened* - in their words, the value of P should be honest - so each dummy variable and spline
term counts. All 25 are therefore counted.

**These criteria are a floor, not a guarantee, for a penalised model.** The 2020 paper notes that
sample size requirements under penalisation methods such as the lasso and elastic net need further
research, because the penalty factors are themselves estimated from the development data and are
often imprecise. The same holds for the ridge penalty used here, whose strength is tuned by
cross-validation on the development data. The calculations below establish that the cohort is not
obviously too small; they do not certify that overfitting has been eliminated. That is what the
bootstrap validation is for.

**The dataset already exists**, so this is not a prospective sample size calculation. The 2020
paper addresses this case directly: with a fixed dataset, the criteria are used to judge whether
the sample is adequate and, by rearrangement, how many predictor parameters can be considered
before overfitting becomes a concern. Section 8 back-solves exactly that.

> The events-per-variable rule of thumb is superseded by this framework and is not used to
> determine adequacy. Riley et al. show the required EPP is context specific: their two worked
> examples require 4.8 and 23 events per parameter respectively.

---
## Setup
Run the two cells below to install dependencies and load libraries.

In [1]:
import numpy as np
import pandas as pd
import pyreadstat

---
## 1. Shared inputs

The following parameters are used across all four criteria. They are defined once here and reused throughout the notebook.

| Parameter | Value | Source |
|-----------|-------|--------|
| φ (phi) | 0.133 | 30-day mortality rate from EDA: 198 / 1,486 |
| N | 1,486 | Patients included after exclusion filter |
| E | 198 | 30-day deaths (observed events) |
| P | 25 | Candidate predictor parameters, read from `config/pipeline.yaml` (19 features; `INTENTIE_cat` contributes 6 dummies and the age spline 2 terms) |
| δ (delta) | 0.05 | Target margin of error and maximum allowed optimism |
| MAPE | 0.05 | Target mean absolute prediction error at 30 days |

Two additional quantities are derived from φ:

- **max(R²_cs):** the theoretical ceiling of the Cox-Snell R², determined entirely by φ. No model can exceed this, regardless of predictor quality.
- **R²_cs conservative:** in the absence of an external estimate, both papers recommend assuming R²_Nagelkerke = 0.15 (Stat Med 2019, Discussion; BMJ 2020, fig 4) — i.e., the model explains 15% of the maximum explainable variation. This translates to R²_cs = 0.15 × max(R²_cs).

In [2]:
import yaml

phi   = 0.133          # 30-day event proportion (EDA: 198 / 1486)
N     = 1486           # patients included after the inclusion filter
E     = 198            # 30-day deaths
delta = 0.05           # target margin of error / maximum allowed optimism
MAPE  = 0.050          # target mean absolute prediction error (BMJ 2020, fig 2; van Smeden et al.)

# P is read from the pipeline config rather than typed in, so this notebook cannot drift
# out of step with the feature set the model actually uses. One parameter per continuous,
# binary and ordinal predictor, plus one per INTENTIE_cat dummy (reference level dropped).
# A splined predictor enters as (knots - 1) basis terms rather than one, and Riley et al.
# count every one of them.
cfg = yaml.safe_load(open("../config/pipeline.yaml", encoding="utf-8"))
feat = cfg["features"]
n_dummies = len(cfg["encoding"]["intentie_categories"]) - 1
n_spline_extra = sum(len(s["knot_quantiles"]) - 2 for s in feat.get("spline", {}).values())
P = sum(len(feat[k]) for k in ("continuous", "binary", "ordinal")) + n_dummies + n_spline_extra
n_features = sum(len(feat[k]) for k in ("continuous", "binary", "ordinal", "categorical"))

# max(R2_cs): the highest Cox-Snell R2 any model can reach at this event rate.
# Formula: Stat Med 2019 eq (23); see also BMJ 2020 supplementary material S5.
max_r2_cs = 1 - (phi ** (2 * phi)) * ((1 - phi) ** (2 * (1 - phi)))

# Conservative anticipated R2_cs. With no external estimate available, Riley et al. suggest
# assuming the model reaches 15% of the maximum explainable variation.
r2_cs_conservative = 0.15 * max_r2_cs

print(f"P (read from config) = {P}   ({n_features} features, of which INTENTIE_cat contributes {n_dummies} dummies; splines add {n_spline_extra} term)")
print(f"max(R2_cs)           = {max_r2_cs:.4f}")
print(f"R2_cs conservative   = {r2_cs_conservative:.4f}  (0.15 x max)")

P (read from config) = 25   (19 features, of which INTENTIE_cat contributes 6 dummies; splines add 1 term)
max(R2_cs)           = 0.5435
R2_cs conservative   = 0.0815  (0.15 x max)


> **Result:** With φ = 0.133, the maximum achievable Cox-Snell R² is **0.54**. The conservative working assumption is R²_cs = **0.082**. The sensitivity analysis (Section 5) shows how required N changes as R²_cs increases toward more realistic values.

---
## 2. Criterion B1 — Precise estimate of the overall outcome rate

**What it tests:** We need enough patients to estimate the 30-day mortality rate (φ) with adequate precision. Specifically, the 95% confidence interval around φ must be no wider than ±δ = ±5 percentage points.

**Formula** (standard binomial proportion sample size):

$$N_{B1} = \left(\frac{1.96}{\delta}\right)^2 \cdot \phi\,(1-\phi)$$

B1 does not depend on the number of predictors P or on R²_cs — it is determined entirely by φ and the precision target δ.

In [3]:
def B1_estimate_outcome_proportion(phi, delta):
    return ((1.96/delta)**2) * (phi*(1-phi))

N_B1 = B1_estimate_outcome_proportion(phi, delta)

print(f"N_B1                 = {N_B1}")

N_B1                 = 177.19149503999998


> **B1 result:** A minimum of **N = 178** patients is needed to estimate φ with 95% CI width ±0.05. Our cohort (N = 1,486) exceeds this by a factor of 8. ✅ **B1 is satisfied.**

---
## 3. Criterion B2 — Small mean absolute prediction error (MAPE)

**What it tests:** The model's predicted probabilities should be accurate on average. The target is MAPE ≤ 0.05 at the 30-day time point, meaning predicted probabilities should be within 5 percentage points of the true probability on average across all patients.

**Formula** (van Smeden et al. 2019 for binary outcomes):

$$N_{B2} = \exp\!\left(\frac{-0.508 + 0.259\,\ln(\phi) + 0.504\,\ln(P) - \ln(\text{MAPE})}{0.544}\right)$$

B2 depends on the event rate (φ), the number of predictors (P), and the MAPE target. It does **not** require an estimate of R²_cs.

In [4]:
def B2_small_mean_error(phi, P, MAPE):
    return np.exp((-0.508+0.259*np.log(phi)+0.504*np.log(P)-np.log(MAPE))/0.544)

N_B2 = B2_small_mean_error(phi, P, MAPE)

print(f"N_B2                 = {N_B2}")

N_B2                 = 731.2373100077793


> **B2 result:** A minimum of **N = 732** patients is needed to achieve MAPE <= 0.05 with
> P = 25 parameters. Our cohort (N = 1,486) satisfies this. Note: the B2 formula is validated by
> simulation only for P <= 30, so P = 25 falls within its valid range. **B2 is satisfied.**

---
## 4. Criterion B3 — Small shrinkage of predictor effects

**What it tests:** When a prediction model is fitted in a finite sample, predictor effect sizes tend to be overestimated — a form of overfitting. B3 requires that the expected uniform shrinkage factor S ≥ 0.9, meaning predictor effects are shrunk by no more than 10% when the model is applied to new data.

**Formula** (BMJ 2020, fig 3; Stat Med 2019, eq 11):

$$N_{B3} = \frac{P}{(S-1)\,\ln\!\left(1 - \dfrac{R^2_{cs}}{S}\right)}$$

B3 is sensitive to both P (more predictors → larger N required) and R²_cs (higher model performance → smaller N required). Because R²_cs is unknown before the model is built, this criterion is the most sensitive to our assumption about model performance.

In [5]:
S_target = 0.9

def n_shrinkage(P, r2_cs, S=0.9):
    return P / ((S - 1) * np.log(1 - r2_cs / S))

N_B3 = n_shrinkage(P, r2_cs_conservative)
print(f"N_B3 = {np.ceil(N_B3):.0f}  (P={P}, R²_cs={r2_cs_conservative:.4f}, S={S_target})")

N_B3 = 2634  (P=25, R²_cs=0.0815, S=0.9)


> **B3 result (conservative):** Under the conservative R2_cs = 0.082, B3 requires **N = 2,634** -
> about 1.8 times our cohort. **B3 is not satisfied under the conservative assumption.** This is
> the most demanding criterion and the one driving the overall required sample size. Whether it is
> satisfied depends on the true R2_cs - see the sensitivity analysis in section 5.

---
## 5. Criterion B4 — Small optimism in apparent model performance

**What it tests:** When a model is evaluated on the same data used to develop it, its apparent R²_Nagelkerke is inflated (optimistic). B4 limits this inflation to δ ≤ 0.05 — i.e., the apparent R² should be within 5% of the true R² on new data.

**Formula** (BMJ 2020, fig 5; Stat Med 2019, eq 26): First derive the shrinkage factor S_B4 implied by the optimism target:

$$S_{B4} = \frac{R^2_{cs}}{R^2_{cs} + \delta \cdot \max(R^2_{cs})}$$

Then apply the same shrinkage formula as B3 using S_B4. B4 is generally less demanding than B3 because S_B4 is close to 1 for models with moderate R²_cs.

In [6]:
def n_optimism(P, r2_cs, max_r2_cs, delta=0.05):
    S_B4 = r2_cs / (r2_cs + delta * max_r2_cs)
    return P / ((S_B4 - 1) * np.log(1 - r2_cs / S_B4))

N_B4 = n_optimism(P, r2_cs_conservative, max_r2_cs)
print(f"N_B4 = {np.ceil(N_B4):.0f}  (P={P}, R²_cs={r2_cs_conservative:.4f}, max={max_r2_cs:.4f})")

N_B4 = 870  (P=25, R²_cs=0.0815, max=0.5435)


> **B4 result (conservative):** Under the conservative R2_cs = 0.082, B4 requires **N = 870**.
> Our cohort (N = 1,486) satisfies this. **B4 is satisfied under the conservative assumption.**

---
## 6. Required N — summary across all criteria

In [7]:
N_required = max(N_B1, N_B2, N_B3, N_B4)
E_required = int(np.ceil(N_required * phi))

criteria = {N_B1: "B1", N_B2: "B2", N_B3: "B3", N_B4: "B4"}
binding = criteria[N_required]

print(f"N_B1 = {np.ceil(N_B1):.0f}  (outcome proportion)")
print(f"N_B2 = {np.ceil(N_B2):.0f}  (MAPE, P={P})")
print(f"N_B3 = {np.ceil(N_B3):.0f}  (shrinkage, P={P})")
print(f"N_B4 = {np.ceil(N_B4):.0f}  (optimism, P={P})")
print()
print(f"N_required  = {np.ceil(N_required):.0f}  (binding criterion: {binding})")
print(f"E_required  = {E_required}")
print()
print(f"Actual N    = {N}")
print(f"Actual E    = {E}")
print(f"N adequate  = {N >= N_required}")
print(f"E adequate  = {E >= E_required}")

N_B1 = 178  (outcome proportion)
N_B2 = 732  (MAPE, P=25)
N_B3 = 2634  (shrinkage, P=25)
N_B4 = 870  (optimism, P=25)

N_required  = 2634  (binding criterion: B3)
E_required  = 351

Actual N    = 1486
Actual E    = 198
N adequate  = False
E adequate  = False


> **Summary:** Under the conservative R²_cs assumption, **B3 (shrinkage) is the binding criterion** — and the only one our cohort fails. B1, B2, and B4 are all satisfied with comfortable margin. The overall adequacy of N = 1,486 hinges entirely on the true R²_cs of the fitted model. The sensitivity analysis below quantifies exactly where the threshold lies.

---
## 7. Sensitivity analysis

The required sample size under B3 and B4 depends on the assumed R²_cs, which is unknown before the model is built. The table below varies R²_cs across four plausible values (0.082–0.20) and three candidate predictor counts (P = 15, 20 and the model's 25).

- **N_B1, N_B2** do not depend on R²_cs and are constant across rows
- **N_B3, N_B4** decrease as R²_cs increases (better model → smaller N required)
- **N_required** = max(B1, B2, B3, B4) — the sample size the study must meet
- 🟢 **N_adequate / E_adequate = True** → our cohort is sufficient
- 🔴 **N_adequate / E_adequate = False** → more data would be needed

In [8]:
r2_values = {
    "Conservative (0.15 × max = 0.082)": r2_cs_conservative,
    "R²_cs = 0.10": 0.10,
    "R²_cs = 0.15": 0.15,
    "R²_cs = 0.20": 0.20,
}
p_values = [15, 20, P]

rows = []
for r2_label, r2 in r2_values.items():
    for p in p_values:
        nb1 = B1_estimate_outcome_proportion(phi, delta)
        nb2 = B2_small_mean_error(phi, p, MAPE)
        nb3 = n_shrinkage(p, r2)
        nb4 = n_optimism(p, r2, max_r2_cs)
        n_req = max(nb1, nb2, nb3, nb4)
        e_req = int(np.ceil(n_req * phi))
        rows.append({
            "R²_cs assumption": r2_label,
            "P": p,
            "N_B1": int(np.ceil(nb1)),
            "N_B2": int(np.ceil(nb2)),
            "N_B3": int(np.ceil(nb3)),
            "N_B4": int(np.ceil(nb4)),
            "N_required": int(np.ceil(n_req)),
            "E_required": e_req,
            "N_adequate": N >= n_req,
            "E_adequate": E >= e_req,
        })

df_results = pd.DataFrame(rows)

def _colour_adequacy(val):
    if val is True:
        return "background-color: #c6efce; color: #276221; font-weight: bold"
    if val is False:
        return "background-color: #ffc7ce; color: #9c0006; font-weight: bold"
    return ""

df_results.style.map(_colour_adequacy, subset=["N_adequate", "E_adequate"])

,R²_cs assumption,P,N_B1,N_B2,N_B3,N_B4,N_required,E_required,N_adequate,E_adequate
0,Conservative (0.15 × max = 0.082),15,178,456,1580,522,1580,211,False,False
1,Conservative (0.15 × max = 0.082),20,178,595,2107,696,2107,281,False,False
2,Conservative (0.15 × max = 0.082),25,178,732,2634,870,2634,351,False,False
3,R²_cs = 0.10,15,178,456,1274,517,1274,170,True,True
4,R²_cs = 0.10,20,178,595,1699,689,1699,226,False,False
5,R²_cs = 0.10,25,178,732,2123,861,2123,283,False,False
6,R²_cs = 0.15,15,178,456,823,502,823,110,True,True
7,R²_cs = 0.15,20,178,595,1097,669,1097,146,True,True
8,R²_cs = 0.15,25,178,732,1372,836,1372,183,True,True
9,R²_cs = 0.20,15,178,456,597,487,597,80,True,True


**Interpretation:**

- **R2_cs = 0.082 (conservative):** inadequate at every predictor count tested. B3 would require
  2,634 patients for P = 25 under this worst case.
- **R2_cs = 0.10:** adequate only for P <= 17; P = 25 remains insufficient.
- **R2_cs = 0.15:** adequate for P = 25 (B3 requires 1,372). This corresponds to a model
  explaining about 28% of the maximum explainable variation in 30-day mortality.
- **R2_cs = 0.20:** comfortably adequate (B3 requires 995).

For context, published prediction models in trauma and surgical critical care typically report
R2_cs in the range of **0.10-0.25**. An R2_cs of at least 0.15 is a reasonable expectation for a
model of 30-day mortality after spinal trauma built from well-established clinical predictors.

---
## 8. Maximum safe number of predictors

As a complement to the table, we back-solve B3 (the binding criterion) to find the **largest P for which N = 1,486 is adequate**, as a function of R²_cs. This directly answers the question: if the model performs at a given R²_cs level, how many predictors can we safely include?

In [9]:
# Back-solve B3 (shrinkage): largest P where N_B3 ≤ N, across r2_cs scenarios
for r2_label, r2 in r2_values.items():
    max_p = max(p for p in range(1, 200) if n_shrinkage(p, r2) <= N)
    print(f"Max P adequate (B3, R²_cs={r2_label}): {max_p:>3}  — P={P} adequate: {max_p >= P}")

Max P adequate (B3, R²_cs=Conservative (0.15 × max = 0.082)):  14  — P=25 adequate: False
Max P adequate (B3, R²_cs=R²_cs = 0.10):  17  — P=25 adequate: False
Max P adequate (B3, R²_cs=R²_cs = 0.15):  27  — P=25 adequate: True
Max P adequate (B3, R²_cs=R²_cs = 0.20):  37  — P=25 adequate: True


**Interpretation:**

- At R2_cs = 0.082 (conservative), the maximum safe predictor count is **14** - below P = 25.
- At R2_cs = 0.10, the limit rises to **17** - still below P = 25.
- At R2_cs = 0.15, the limit reaches **27**, so P = 25 falls inside the safe range.
- At R2_cs = 0.20, the limit is **37** - ample headroom.

The critical threshold lies between R2_cs = 0.10 and 0.15. If the fitted model achieves
R2_cs >= 0.15, proceeding with all 25 parameters is justified.

---
## 9. Conclusion

We assessed whether N = 1,486 / E = 198 is adequate to develop a binary prediction model with
**P = 25 candidate predictor parameters** (19 features; `INTENTIE_cat` contributes 6 dummies
and the age spline 2 terms).

**Results by criterion:**

| Criterion | Required N (conservative, R2_cs = 0.082) | Required N (R2_cs = 0.15) | Cohort adequate? |
|---|---|---|---|
| B1 - outcome rate precision | 178 | 178 | Yes |
| B2 - MAPE <= 0.05 | 732 | 732 | Yes |
| B3 - shrinkage <= 10% | 2,634 | 1,372 | No / Yes (depends on R2_cs) |
| B4 - optimism <= 0.05 | 870 | 836 | Yes |

**Binding criterion:** B3 (shrinkage) drives the required sample size in every scenario. B1, B2
and B4 are comfortably satisfied even under the conservative assumption.

### Resolution against the observed model

The conservative assumption above (R2_cs = 0.082) is a placeholder used because no estimate was
available in advance. The pilot model in the EDA notebook provides one: its **cross-validated
R2_cs is 0.243**, well above the 0.15 threshold this analysis identified as the critical value.

At R2_cs = 0.243, B3 requires **N = 795** and permits up to **46 parameters**. With N = 1,486 and
P = 25, the cohort is adequate with a wide margin, and the conditional above is resolved in
favour of proceeding.

> This is a *check*, not an a priori justification: the R2_cs used here was estimated from the
> same data whose adequacy is being assessed, so it cannot serve as independent evidence. It is
> reported because a value far above the assumed threshold is reassuring, and one far below it
> would have required reducing the predictor set. Cross-validation rather than the apparent value
> is used to limit the optimism in that check.

**Decision:** proceed with all 25 parameters.

> The events-per-parameter heuristic (>= 10 events per parameter, i.e. 250 events here) is not
> used to determine adequacy. The Riley framework supersedes it precisely because the rule of
> thumb ignores how much signal the predictors carry; at the observed R2_cs this cohort supports
> considerably more parameters than the heuristic would allow.

---
## 10. Cross-validation and internal validation strategy

| Scheme | E_train (approx) | E_test per fold (approx) | Assessment |
|--------|-----------------|--------------------------|------------|
| 5-fold CV | 198 × 4/5 = **158** | **40** | Acceptable — ≥25 test events per fold, AUROC stable |
| 10-fold CV | 198 × 9/10 = **178** | **20** | Marginal — AUROC per fold unreliable at n = 20 events |

**Recommendation: 5-fold cross-validation** for hyperparameter tuning during model development.

For **internal validation**, Riley et al. recommend bootstrap rather than data splitting. Bootstrap internal validation directly estimates the optimism in apparent model performance — the same quantity controlled by B4 — without the additional variance penalty of a held-out test fold. The Phase 4 plan uses **n = 1,000 bootstrap iterations** to derive optimism-corrected AUROC (C-statistic) and calibration estimates. This approach is consistent with B4's objective and makes efficient use of all 1,486 patients.